# Human in the loop (HITL) for Agentic systems
---

Agentic systems are revolutionizing how businesses automate complex workflows and decision-making processes. Building systems that are deterministic and reliable, while automating user tasks is essential, but at the end of the day, these systems are built on models that are non-deterministic.

Building intelligent autonomous agents that effectively handle user queries requires careful planning and robust safeguards. Although FMs continue to improve, they can still produce incorrect outputs, and because agents are complex systems, errors can occur at multiple stages. For example, an agent might select the wrong tool or use correct tools with incorrect parameters. This might lead to the agent causing latency and cost to your application.

In sensitive scenarios, **human-in-the-loop** (`HITL`) interaction is essential for successful AI agent deployments, encompassing multiple critical touchpoints between humans and automated systems. `HITL` can take many forms, from end-users approving actions and providing feedback, to subject matter experts reviewing responses offline and agents working alongside customer service representatives. The common thread is maintaining human oversight and using human intelligence to improve agent performance. This human involvement helps establish ground truth, validates agent responses before they go live, and enables continuous learning through feedback loops.

## How `HITL` works in LangGraph?
---

LangGraph supports a robust human in the loop mechanism in agent workflows. This means that the framework enables humans or experts to intervene at any point in an automated process. 

### Key capabilities

1. ***Persistent execution state***: `LangGraph` checkpoints the graph state after each step, allowing execution to pause indefinitely at defined nodes. This supports  asynchronous human review or input without time constraints.

2. ***Flexible integration points***: `HITL` logic can be introduced at any point in the workflow. This allows targeted human involvement, such as approving API calls, correcting outputs, or guiding conversations.

### Typical use cases

1. ***🛠️ Reviewing tool calls***: Humans can review, edit, or approve tool calls requested by the LLM before tool execution.
2. ***✅ Validating LLM outputs***: Humans can review, edit, or approve content generated by the LLM.
3. ***💡 Providing context***: Enable the LLM to explicitly request human input for clarification or additional details or to support multi-turn conversations.

### Use case
---

Let's build a product agent that can list some of the high priority tasks and also update the tasks. We will be using a human in the loop to check for the priority, review it, and review the task creation/updating process. We will do these using three ways: 

1. By `interrupting` the entire graph execution, 

2. We will use the `command` primitive to resume the execution with a value provided by the human and

3. We will build a custom router that will integrate a human in the loop implementation in a highly flexible and customized manner.

These three ways can be used in a hierarchical manner depending on the level of control you want the human to have or developers to have on the agentic system.

In [1]:
# First, build the simple agent that has access to local tools
import logging

# first, lets import the necessary libraries required to build the agent in this notebook
from typing import TypedDict, Annotated, List, Optional, Dict, Any
from langgraph.graph import StateGraph, END, MessagesState
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables.graph import MermaidDrawMethod
from IPython.display import Image, display
from pydantic import BaseModel

In [2]:
# set a logger
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

### Step 1: Define the Agent State

We will first define the agent state that will remain throughout the operation. This is the state of the graph and the state schema serves as the input for all nodes and edges in the graph.

In [16]:
from typing import Dict, Any, List, Optional
from pydantic import BaseModel

# Next, we will us the task schema in the graph schema.

# This is the schema that will be used and updated dynamically across graph execution
# which includes the task, the summary and the active campaign information
# Define your State schema
class WorkflowState(BaseModel):
    """Schema for the entire workflow state"""
    user_message: str
    tasks: List[Dict[str, Any]] = []
    messages: List[Any] = []
    summary: Optional[str] = None
    num_tasks_completed: Optional[int] = None
    num_tasks_to_do: Optional[int] = None
    num_tasks_in_progress: Optional[int] = None
    num_tasks_critical: Optional[int] = None
    response: Optional[str] = None

In [17]:
import boto3
session = boto3.session.Session()
region = session.region_name
logger.info(f"Running this example in region: {region}")

# Initialize the bedrock client placeholder
bedrock_client = boto3.client("bedrock-runtime")


# represents the global variables used across this notebook
BEDROCK_RUNTIME: str = 'bedrock-runtime'
# Model ID used by the agent
AMAZON_NOVA_LITE_MODEL_ID: str = "us.amazon.nova-lite-v1:0"
PROVIDER_ID : str = 'amazon'
# Inference parameters
TEMPERATURE: float = 0.1
MAX_TOKENS: int = 512

[2025-05-14 17:53:26,721] p5460 {65438041.py:4} INFO - Running this example in region: us-east-1


In [18]:
import boto3
import logging
# We are importing this to use any model supported on Amazon Bedrock. In this example
# we will be using the Amazon Nova lite model.
from langchain_aws import ChatBedrockConverse
# This helps checkpoint the memory state of the agent for short term/long term memory
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables.config import RunnableConfig
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create the llm used by the agent
llm = ChatBedrockConverse(
    model=AMAZON_NOVA_LITE_MODEL_ID,
    provider=PROVIDER_ID, 
    temperature=TEMPERATURE, 
    max_tokens=MAX_TOKENS,
    client=bedrock_client,
)
logger.info(f"Initialized the model {llm} that will power our agent.")

[2025-05-14 17:53:27,068] p5460 {2282892260.py:19} INFO - Initialized the model client=<botocore.client.BedrockRuntime object at 0x1225279b0> model_id='us.amazon.nova-lite-v1:0' max_tokens=512 temperature=0.1 provider='amazon' supports_tool_choice_values=['auto'] that will power our agent.


### Define tools
---

Next, we will define some tools that our langGraph agent will use. This includes listing the tasks, getting the tasks by id, updating the task, getting the task summary, creating a task, and recommending some next steps.

In [19]:
import os
import json
# represents the location of the synthetic data. Ideally
# this would be an API call from a service like JIRA, but in this case
# the data is a JSON file which is synthetically generated using Claude 3.7 Sonnet on Amazon Bedrock
DATA_DIR: str = "data"
SYNTHETIC_TASKS_FNAME: str = "team_tasks.json"
TASKS_DATA_FPATH: str = os.path.join("..", DATA_DIR, SYNTHETIC_TASKS_FNAME)

In [20]:
logger.info(f"Going to use synthetic data from the following directory -> {TASKS_DATA_FPATH}. (This would ideally be an API call)")

[2025-05-14 17:53:28,413] p5460 {3891557946.py:1} INFO - Going to use synthetic data from the following directory -> ../data/team_tasks.json. (This would ideally be an API call)


In [21]:
from langchain_core.tools import tool

@tool
def list_tasks(status: Optional[str] = None, priority: Optional[str] = None) -> Dict[str, any]:
    """
    List the tasks with an optional filtering by status or priority. This tool is used 
    when the user asks a question about listing the tasks. This tool lists all the tasks and if 
    the user provides some filtering criteria for status or the priority of tasks to be listed, then
    the agent uses that to list some of the tasks.
    
    Args:
        status: Optional filter for task status (to-do, in-progress, completed, at-risk)
        priority: Optional filter for task priority (low, medium, high)
    
    Returns:
        Dictionary with filtered tasks
    """
    with open(TASKS_DATA_FPATH, 'r') as f:
        data = json.load(f)
    filtered_tasks = data['tasks']
    # Filter by status if needed
    if status:
        filtered_tasks = [task for task in filtered_tasks if task['status'] == status]
    # Filter by priority if needed
    if priority:
        filtered_tasks = [task for task in filtered_tasks if task['priority'] == priority]
    return {"tasks": filtered_tasks}

In [22]:
@tool
def update_task_status(task_id: str, new_status: str) -> Dict[str, Any]:
    """
    Update the status of a task.
    
    Args:
        task_id: The unique identifier of the task
        new_status: The new status to set (to-do, in-progress, completed, at-risk)
    
    Returns:
        Dictionary with updated task or error message
    """
    valid_statuses = ["to-do", "in-progress", "completed", "at-risk"]
    if new_status not in valid_statuses:
        return {"error": f"Invalid status. Must be one of {valid_statuses}"}
    
    with open(TASKS_DATA_FPATH, 'r') as f:
        data = json.load(f)
    
    for i, task in enumerate(data['tasks']):
        if task['id'] == task_id:
            data['tasks'][i]['status'] = new_status
            
            # Write back to file
            with open(TASKS_DATA_FPATH, 'w') as f:
                json.dump(data, f, indent=2)
            
            return {"task": data['tasks'][i], "message": f"Updated task {task_id} status to {new_status}"}
    
    return {"error": f"Task with ID {task_id} not found"}


In [23]:
@tool
def create_task(name: str, description: str, due_date: str, priority: str) -> Dict[str, Any]:
    """
    Create a new task with the specified details.
    
    Args:
        name: The name of the task
        description: Detailed description of the task
        due_date: Due date in YYYY-MM-DD format
        priority: Priority level (low, medium, high)
    
    Returns:
        Dictionary with the created task information
    """
    valid_priorities = ["low", "medium", "high"]
    if priority not in valid_priorities:
        return {"error": f"Invalid priority. Must be one of {valid_priorities}"}
    
    # Validate date format
    try:
        datetime.strptime(due_date, '%Y-%m-%d')
    except ValueError:
        return {"error": "Invalid date format. Use YYYY-MM-DD"}
    
    with open(TASKS_DATA_FPATH, 'r') as f:
        data = json.load(f)
    
    # Generate new ID
    task_ids = [int(task['id'].split('-')[1]) for task in data['tasks']]
    new_id = max(task_ids) + 1
    new_id_str = f"TASK-{new_id:03d}"
    
    new_task = {
        "id": new_id_str,
        "name": name,
        "description": description,
        "due_date": due_date,
        "status": "to-do",
        "priority": priority
    }
    
    data['tasks'].append(new_task)
    
    # Write back to file
    with open(TASKS_DATA_FPATH, 'w') as f:
        json.dump(data, f, indent=2)
    
    return {"task": new_task, "message": f"Created new task with ID {new_id_str}"}


## Using Interrupt
---

Use `interrupt` at `safety-critical` branches where no automated fallback is acceptable. Execution will pause at that node until a human operator invokes the resume command.

In [67]:
from langgraph.types import interrupt, Command

@tool
def human_approve_changes(prompt: str) -> dict:
    """
    A dummy tool that pauses execution for human approval.
    When called, the graph will interrupt at this point
    until someone calls `resume` on the workflow.
    """
    # This will halt the graph at this spot,
    # showing the given prompt to the human operator.
    interrupt(
        prompt=prompt,
        resume_node="apply_changes"   # name of the node to jump to once resumed
    )
    return {"approved": True}

In [68]:
# define the tools that the ReAct agent can utilize
tools = [list_tasks, update_task_status, create_task, human_approve_changes]

In [69]:
SYSTEM_PROMPT = """You are Maya, a Project Manager Assistant designed to help manage LangGraph development tasks.
You have access to the team's tasks and can perform various operations to help the team stay organized.

Your goal is to help the user understand the current state of their project, provide relevant information about tasks,
and assist with task management.

Analyze the user's request carefully and choose the appropriate tools to fulfill their needs.
Use a step-by-step approach to complex requests, and always provide a helpful summary of your actions.

You have access to the following tools based on the user question and can invoke them individually or sequentially based on the user
request: list_tasks, update_task_status, create_task.

Use the human_approve_changes tool if there is any question about updating a task or creating a task.
"""

In [70]:
# Import the function to create a prebuilt ReAct agent from LangGraph.
from langgraph.prebuilt import create_react_agent

# Create the ReAct agent by binding the language model (llm) with the defined tools.
# This agent can now reason about when to use these tools based on user input.
get_realtime_info_react_llm = create_react_agent(
    llm,
    tools=tools, 
    prompt=SYSTEM_PROMPT
)

### Create graph nodes
---

In this portion of the notebook, we create two nodes, one that uses the task schema to process the user input, updates with the updated state after the user inputs the request.

In [71]:
# Define the first node that processes user input
def process_user_input(state: WorkflowState) -> WorkflowState:
    """
    This function processes the user's message and initializes the state.
    """
    # Load task data to populate the state
    try:
        with open(TASKS_DATA_FPATH, 'r') as f:
            data = json.load(f)
        
        # Calculate metrics
        num_completed = sum(1 for task in data['tasks'] if task["status"] == "completed")
        num_to_do = sum(1 for task in data['tasks'] if task["status"] == "to-do")
        num_in_progress = sum(1 for task in data['tasks'] if task["status"] == "in-progress")
        num_critical = sum(1 for task in data['tasks'] if task["status"] == "at-risk")
        
        # Create the summary
        summary = f"Project Status: {len(data['tasks'])} total tasks, {num_completed} completed, {num_to_do} to-do, {num_in_progress} in progress, {num_critical} critical"
        
        # Return a new state object with all the data
        return WorkflowState(
            user_message=state.user_message,
            tasks=data['tasks'],
            messages=[],
            summary=summary,
            num_tasks_completed=num_completed,
            num_tasks_to_do=num_to_do,
            num_tasks_in_progress=num_in_progress,
            num_tasks_critical=num_critical
        )
    except Exception as e:
        logger.error(f"Error loading task data: {e}")
        return WorkflowState(
            user_message=state.user_message,
            messages=[]
        )

In [72]:
def process_task_request(state: WorkflowState) -> WorkflowState:
    """
    This function processes the task request through the ReAct agent.
    """
    try:
        # Create the message for the agent
        messages = [
            HumanMessage(content=state.user_message)
        ]
        
        # Add context about the tasks to the message
        task_context = f"\nCurrent Project Status: {state.summary or 'No summary available'}"
        messages[0].content += task_context
        
        agent_input = {
            "messages": messages
        }
        
        # Invoke the agent
        result = get_realtime_info_react_llm.invoke(agent_input)
        
        # Extract the output
        if isinstance(result, dict) and "output" in result:
            response = result["output"]
        else:
            response = str(result)
        
        # Update the conversation history
        updated_messages = [
            *state.messages,
            HumanMessage(content=state.user_message),
            AIMessage(content=response)
        ]
        
        # Create a dictionary from the current state
        state_dict = state.model_dump()
        
        # Update the specific fields we want to change
        state_dict["messages"] = updated_messages
        state_dict["response"] = response
        
        # Return a new state with updated values
        return WorkflowState(**state_dict)
    except Exception as e:
        logger.error(f"Error processing task request: {e}")
        error_message = f"I apologize, but I encountered an error while processing your request: {str(e)}"
        
        # Print the exception for debugging
        import traceback
        traceback.print_exc()
        
        # Create a dictionary from the current state
        state_dict = state.model_dump()
        
        # Update with error information
        state_dict["messages"] = [
            *state.messages,
            HumanMessage(content=state.user_message),
            AIMessage(content=error_message)
        ]
        state_dict["response"] = error_message
        
        # Return updated state with error
        return WorkflowState(**state_dict)

In [73]:
def print_stream(result):
    """
    Pretty prints the raw response from a LangGraph workflow result.
    """
    import json
    from pprint import pprint
    
    print("\n" + "=" * 50 + "\n")
    
    # For dictionary results
    if isinstance(result, dict):
        if "messages" in result:
            # Print each message in a readable format
            for i, message in enumerate(result["messages"]):
                # Get message type
                msg_type = type(message).__name__
                print(f"[Message {i+1}] Type: {msg_type}")
                
                # Print content in readable format
                if hasattr(message, "content"):
                    print("\nContent:")
                    if isinstance(message.content, list):
                        for j, part in enumerate(message.content):
                            print(f"\n-- Part {j+1} --")
                            pprint(part)
                    else:
                        print(message.content)
                    
                print("\n" + "-" * 50 + "\n")
        else:
            # Just pretty print the dictionary
            pprint(result)
    
    # For list results
    elif isinstance(result, list):
        for i, item in enumerate(result):
            print(f"\n[Item {i+1}]\n")
            print_stream(item)  # Recursively handle items
    
    # For other types
    else:
        print(f"Type: {type(result)}")
        print("\nContent:")
        print(result)
        
    print("\n" + "=" * 50 + "\n")

In [74]:
# 1) Add the approval node
def check_and_interrupt(state: WorkflowState) -> WorkflowState:
    if state.num_tasks_critical and state.num_tasks_critical > 0:
        # invoke your new tool to trigger the interrupt
        human_approve_changes(
            prompt="⚠️ You have critical tasks pending. Approve proceeding with updates?"
        )
    return state

def apply_changes(state: WorkflowState) -> WorkflowState:
    """
    This node runs after the human-in-the-loop approval.
    It marks all currently “critical” tasks as cleared and adds
    a confirmation message into the state.
    """
    # Find and update any critical tasks
    for task in state.tasks:
        if task.get("status") == "at-risk":
            task["status"] = "in-progress"
    
    # Add a confirmation into the state’s response
    state.response = (
        "✅ All critical tasks have been acknowledged and updated. "
        "Proceeding with the workflow."
    )
    
    # Optionally bump your metrics
    state.num_tasks_in_progress = sum(1 for t in state.tasks if t["status"] == "in-progress")
    state.num_tasks_critical   = sum(1 for t in state.tasks if t["status"] == "at-risk")
    
    return state

In [75]:
workflow = StateGraph(WorkflowState)
workflow.add_node("process_user_input",    process_user_input)
workflow.add_node("process_task_request",  process_task_request)
workflow.add_node("check_and_interrupt",   check_and_interrupt)
workflow.add_node("apply_changes",         apply_changes)

# 4) Wire up the edges
workflow.set_entry_point("process_user_input")
workflow.add_edge("process_user_input",   "process_task_request")
workflow.add_edge("process_task_request", "check_and_interrupt")
workflow.add_edge("check_and_interrupt",  "apply_changes")
workflow.add_edge("apply_changes",        END)

compiled = workflow.compile(checkpointer=MemorySaver())

In [76]:
config = {"configurable": {"thread_id": "thread_1"}}
input_data = {"user_message": "Can you list all of my tasks?"}
result = compiled.invoke(input_data, config=config)
print_stream(result)

[2025-05-14 20:06:45,610] p5460 {bedrock_converse.py:610} INFO - Using Bedrock Converse API to generate response
[2025-05-14 20:06:46,652] p5460 {bedrock_converse.py:610} INFO - Using Bedrock Converse API to generate response




[Message 1] Type: HumanMessage

Content:
Can you list all of my tasks?

--------------------------------------------------

[Message 2] Type: AIMessage

Content:
{'messages': [HumanMessage(content='Can you list all of my tasks?\nCurrent Project Status: Project Status: 36 total tasks, 5 completed, 30 to-do, 1 in progress, 0 critical', additional_kwargs={}, response_metadata={}, id='84652724-bd64-4334-87f9-98a847e89cd7'), AIMessage(content=[{'type': 'text', 'text': '<thinking>The user has asked to list all of their tasks. I will use the `list_tasks` tool to retrieve the tasks without any filtering criteria.</thinking>\n'}, {'type': 'tool_use', 'name': 'list_tasks', 'input': {}, 'id': 'tooluse_TtlzhYalQNmg50Q_QRsQTA'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '5deddd1d-b296-4d81-b559-6d7e272467c7', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 15 May 2025 00:06:46 GMT', 'content-type': 'application/json', 'content-length': '431', 'connection': '

In [77]:
config = {"configurable": {"thread_id": "thread_1"}}
input_data = {"user_message": "Can you the first task to completed?"}
result = compiled.invoke(input_data, config=config)
print_stream(result)

[2025-05-14 20:07:05,901] p5460 {bedrock_converse.py:610} INFO - Using Bedrock Converse API to generate response


[2025-05-14 20:07:06,436] p5460 {bedrock_converse.py:610} INFO - Using Bedrock Converse API to generate response
[2025-05-14 20:07:07,220] p5460 {bedrock_converse.py:610} INFO - Using Bedrock Converse API to generate response




[Message 1] Type: HumanMessage

Content:
Can you the first task to completed?

--------------------------------------------------

[Message 2] Type: AIMessage

Content:
{'messages': [HumanMessage(content='Can you the first task to completed?\nCurrent Project Status: Project Status: 36 total tasks, 5 completed, 30 to-do, 1 in progress, 0 critical', additional_kwargs={}, response_metadata={}, id='c7d87ab1-4fae-4534-ae53-9422cc48b467'), AIMessage(content=[{'type': 'text', 'text': "<thinking>The user wants to mark the first task as completed. To do this, I need to identify the first task and then update its status to 'completed'. I will first list the tasks to identify the first task's ID.</thinking>\n"}, {'type': 'tool_use', 'name': 'list_tasks', 'input': {'status': 'to-do'}, 'id': 'tooluse_JRssTudVTVSLmEPCeH8h2Q'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '75b31590-11a4-4b9e-808e-b908a2514f14', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 15 M